# CyberBullying Detector — DistilBERT (Colab T4)
**Run cells top to bottom. Do not skip any cell.**

In [ ]:
# ── CELL 1: GPU check ─────────────────────────────────────────
import torch
assert torch.cuda.is_available(), "No GPU! Go to Runtime → Change runtime type → T4 GPU"
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── CELL 2: Mount Drive ────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, sys

PROJECT_DIR = '/content/drive/MyDrive/CyberBullying_2.0'
SRC_DIR     = os.path.join(PROJECT_DIR, 'src')

assert os.path.isdir(PROJECT_DIR), f"Not found: {PROJECT_DIR}"
assert os.path.isdir(SRC_DIR),     f"Not found: {SRC_DIR}"

# Add src/ first so its data_loader / bert_trainer take priority
for p in [SRC_DIR, PROJECT_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

os.chdir(PROJECT_DIR)
print("Working dir :", os.getcwd())
print("sys.path[0] :", sys.path[0])

In [ ]:
# ── CELL 3: Install dependencies ──────────────────────────────
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.40.0', 'accelerate>=0.24.0', 'pyyaml', 'scikit-learn', 'openpyxl'])
print("Done.")

In [ ]:
# ── CELL 4: Config ─────────────────────────────────────────────
import os, sys

PROJECT_DIR = '/content/drive/MyDrive/CyberBullying_2.0'
SRC_DIR     = os.path.join(PROJECT_DIR, 'src')
os.chdir(PROJECT_DIR)

CONFIG = dict(
    model_name              = 'distilbert-base-uncased',
    max_length              = 128,
    batch_size              = 32,
    num_epochs              = 3,
    learning_rate           = 2e-5,
    weight_decay            = 0.01,
    warmup_ratio            = 0.06,
    early_stopping_patience = 2,
)
SAMPLE_FRAC = 1.0
SAVE_DIR    = os.path.join(PROJECT_DIR, 'models', 'distilbert_colab')
os.makedirs(SAVE_DIR, exist_ok=True)

print("Config  :", CONFIG)
print("Save dir:", SAVE_DIR)

In [ ]:
# ── CELL 5: Load & split data ─────────────────────────────────
import os, re, importlib.util, pathlib, pickle, gc
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

PROJECT_DIR = '/content/drive/MyDrive/CyberBullying_2.0'
SRC_DIR     = os.path.join(PROJECT_DIR, 'src')
SAMPLE_FRAC = 1.0
SPLITS_PATH = '/content/splits.pkl'

os.chdir(PROJECT_DIR)

spec = importlib.util.spec_from_file_location(
    'data_loader', pathlib.Path(SRC_DIR) / 'data_loader.py'
)
dl = importlib.util.module_from_spec(spec)
spec.loader.exec_module(dl)
load_all_datasets = dl.load_all_datasets

def clean(text):
    text = str(text).lower()
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'@\w+', ' ', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    return re.sub(r'\s+', ' ', text).strip()

print("Loading datasets...")
df = load_all_datasets(os.path.join(PROJECT_DIR, 'config', 'datasets.yaml'))

if SAMPLE_FRAC < 1.0:
    df = df.groupby('label', group_keys=False).apply(
        lambda g: g.sample(frac=SAMPLE_FRAC, random_state=42)
    ).reset_index(drop=True)

print("Cleaning text...")
df['text'] = df['text'].apply(clean)
df = df[df['text'].str.len() > 3].reset_index(drop=True)
print(f"After cleaning : {len(df):,} rows")
print(f"Safe (0)       : {(df.label==0).sum():,}")
print(f"Bullying (1)   : {(df.label==1).sum():,}")

X, y = df['text'].tolist(), df['label'].tolist()
del df; gc.collect()   # free ~1 GB before splitting

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val,   X_test, y_val,   y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)
del X, y, X_temp, y_temp; gc.collect()   # keep only the 3 splits

print(f"Train: {len(X_train):,}  Val: {len(X_val):,}  Test: {len(X_test):,}")

with open(SPLITS_PATH, 'wb') as f:
    pickle.dump((X_train, y_train, X_val, y_val, X_test, y_test), f)
print(f"Splits saved to {SPLITS_PATH}")

In [ ]:
# ── CELL 6: Train DistilBERT ───────────────────────────────────
import os, importlib.util, pathlib, pickle

PROJECT_DIR = '/content/drive/MyDrive/CyberBullying_2.0'
SRC_DIR     = os.path.join(PROJECT_DIR, 'src')
SAVE_DIR    = os.path.join(PROJECT_DIR, 'models', 'distilbert_colab')
SPLITS_PATH = '/content/splits.pkl'

CONFIG = dict(
    model_name              = 'distilbert-base-uncased',
    max_length              = 128,
    batch_size              = 32,
    num_epochs              = 3,
    learning_rate           = 2e-5,
    weight_decay            = 0.01,
    warmup_ratio            = 0.06,
    early_stopping_patience = 2,
)

# Load splits — works whether X_train is in memory or only on disk
try:
    X_train, y_train, X_val, y_val, X_test, y_test
    print("Using splits already in memory.")
except NameError:
    print(f"Reloading splits from {SPLITS_PATH} ...")
    with open(SPLITS_PATH, 'rb') as f:
        X_train, y_train, X_val, y_val, X_test, y_test = pickle.load(f)
    print(f"Train: {len(X_train):,}  Val: {len(X_val):,}  Test: {len(X_test):,}")

spec = importlib.util.spec_from_file_location(
    'bert_trainer', pathlib.Path(SRC_DIR) / 'bert_trainer.py'
)
bt = importlib.util.module_from_spec(spec)
spec.loader.exec_module(bt)

os.makedirs(SAVE_DIR, exist_ok=True)
model, tokenizer = bt.train(
    X_train, y_train,
    X_val,   y_val,
    X_test,  y_test,
    save_dir=pathlib.Path(SAVE_DIR),
    **CONFIG,
)

In [ ]:
# ── CELL 7: Smoke test ─────────────────────────────────────────
import os, importlib.util, pathlib

PROJECT_DIR = '/content/drive/MyDrive/CyberBullying_2.0'
SRC_DIR     = os.path.join(PROJECT_DIR, 'src')
SAVE_DIR    = os.path.join(PROJECT_DIR, 'models', 'distilbert_colab')

spec = importlib.util.spec_from_file_location(
    'bert_trainer', pathlib.Path(SRC_DIR) / 'bert_trainer.py'
)
bt = importlib.util.module_from_spec(spec)
spec.loader.exec_module(bt)

predictor = bt.BertPredictor(os.path.join(SAVE_DIR, 'distilbert_cyberbullying'))

for text in [
    "you are so stupid and worthless",
    "hey, hope you have a great day!",
    "nobody likes you, go kill yourself",
    "good game everyone, well played",
]:
    r = predictor.predict(text)
    print(f"[{r['label']:8s} {r['prob_bully']:.2f}]  {text}")

## Done
Model saved to `models/distilbert_colab/distilbert_cyberbullying/` on your Drive.
Download that folder to your local `models/` directory to use with `predict.py`.